In [1]:
from pathlib import Path

from pypdf import PdfReader
from langchain_core.documents import Document


def load_pdfs(data_dir: str) -> list[Document]:
    """Recursively load all PDFs from a directory."""

    documents = []

    for pdf_path in Path(data_dir).rglob("*.pdf"):
        print(f"Loading {pdf_path.name}")

        reader = PdfReader(str(pdf_path))

        for page_num, page in enumerate(reader.pages):
            text = page.extract_text() or ""

            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": str(pdf_path),
                        "paper": pdf_path.stem,
                        "topic": pdf_path.parent.name,
                        "page": page_num + 1,
                    },
                )
            )

    return documents


docs = load_pdfs("../data/papers")

print(f"Loaded {len(docs)} pages.")
print(f"Total characters: {sum(len(doc.page_content) for doc in docs):,}")

Loading roberta.pdf
Loading review_of_bert_based_models.pdf
Loading bert_review.pdf
Loading distil_bert.pdf
Loading what_the_mask.pdf
Loading attention_is_all_you_need.pdf
Loading generative_adversarial_transformers.pdf
Loading transformer_in_transformer.pdf
Loading reformer.pdf
Loading huggingface_transformers.pdf
Loading time_series_transformer.pdf
Loading generative_pretrained_transformer.pdf
Loading gptq.pdf
Loading graph_gpt.pdf
Loading bio_gpt.pdf
Loading graph_based_reranking.pdf
Loading dense_text_retrieval.pdf
Loading colbert.pdf
Loading improve_recall_for_document_retrieval_syste,s.pdf
Loading sparse_dense_and_attentional_representation_for_text_retrieval.pdf
Loading best_practices_in_Rag.pdf
Loading light_rag.pdf


Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)


Loading rag_for_knowledge_intensive_tasks.pdf
Loading corag.pdf
Loading graph_rag.pdf
Loaded 461 pages.
Total characters: 1,889,060


In [2]:
len(docs)

461

In [3]:
docs[2].metadata

{'source': '../data/papers/bert/roberta.pdf',
 'paper': 'roberta',
 'topic': 'bert',
 'page': 3}

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    add_start_index=True,
)

# Split the documents into chunks
chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks from {len(docs)} pages.")

/workspaces/nlp-paper-rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created 2896 chunks from 461 pages.


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights: 100%|██████████| 103/103 [00:02<00:00, 44.91it/s]


In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
document_ids = vector_store.add_documents(chunks)

print(f"Stored {len(document_ids)} chunks.")

Stored 2896 chunks.


In [8]:
results = vector_store.similarity_search(
    "What is retrieval augmented generation?",
    k=3
)

In [9]:
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}")
    print("-" * 80)
    print(doc.page_content[:500])


Result 1
--------------------------------------------------------------------------------
ter context for retrieval-augmented generation. arXiv
preprint arXiv:2311.08377.
Shitao Xiao, Zheng Liu, Peitian Zhang, and Niklas
Muennighoff. 2023. C-pack: Packaged resources
to advance general chinese embedding. Preprint,
arXiv:2309.07597.
17727

Result 2
--------------------------------------------------------------------------------
and the quality of retrieval is comparable to generation. The result of PRO2RET(Need retrieval) is better than
PRO2RET(Need generation) demonstrating that expanding the size of the retrieval sources can improve outcomes
effectively.
Result of retrieval Result of generationPrompt
A family
A tyrannosaurus 
Figure 5: Some cases of retrieval and generation methods: the generation model is less controllable, occasionally
producing errors or low-quality outputs. On the contrary, since the retrieval so

Result 3
----------------------------------------------------------

In [10]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert assistant answering questions about NLP research papers.

Use ONLY the context below.

If the answer cannot be found in the context, reply:

"I don't know."

Context:
{context}

Question:
{input}
""")

In [19]:
# Create retriever from vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Create the RAG chain using LCEL
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: retriever.invoke(x["input"])
    )
    | prompt
    | llm
)

In [21]:
response = rag_chain.invoke({"input": "What is Data networks"})

# The response is an AIMessage, so access the content directly
print(response.content)

I don't know.


In [22]:
import json
import random
from tqdm import tqdm

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Load benchmark
with open("../data/questions/benchmark.json") as f:
    benchmark = json.load(f)

print(f"Total questions in benchmark: {len(benchmark)}")

# Shuffle with seed for reproducibility
shuffled_benchmark = benchmark.copy()
random.shuffle(shuffled_benchmark)

# Select 50 questions
sample_size = 50
sample_benchmark = shuffled_benchmark[:sample_size]

print(f"Evaluating on {len(sample_benchmark)} shuffled questions (seed={RANDOM_SEED})")
print(f"Sample IDs: {[q['id'] for q in sample_benchmark[:3]]}...")  # Show first 3 IDs

Total questions in benchmark: 500
Evaluating on 50 shuffled questions (seed=42)
Sample IDs: ['generative_adversarial_transformers_q005', 'huggingface_transformers_q004', 'bert_review_q006']...


In [23]:
import time

# Run RAG pipeline on sampled questions
print("Running RAG pipeline on 50 questions...")
evaluation_data = []
failed_questions = []

for i, item in enumerate(tqdm(sample_benchmark)):
    try:
        # Get retrieved documents
        retrieved_docs = retriever.invoke(item["question"])
        contexts = [doc.page_content for doc in retrieved_docs]
        
        # Get RAG response
        response = rag_chain.invoke({"input": item["question"]})
        response_text = response.content if hasattr(response, 'content') else str(response)
        
        # Collect evaluation data
        evaluation_data.append({
            "user_input": item["question"],
            "retrieved_contexts": contexts,
            "response": response_text,
            "reference": item["ground_truth"],
            "id": item["id"],
        })
        
    except Exception as e:
        print(f"Error on {item['id']}: {e}")
        failed_questions.append(item["id"])
        continue
    
    # Rate limiting to avoid API throttling
    if (i + 1) % 10 == 0:
        time.sleep(0.5)

print(f"\nProcessed {len(evaluation_data)} questions successfully")
if failed_questions:
    print(f"Failed on {len(failed_questions)} questions: {failed_questions[:5]}...")

Running RAG pipeline on 50 questions...


 14%|█▍        | 7/50 [00:11<01:51,  2.60s/it]

Error on rag_for_knowledge_intensive_tasks_q016: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kg2cw9b2efxa2n263c6qe19r` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4862, Requested 1176. Please try again in 380ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


100%|██████████| 50/50 [06:38<00:00,  7.97s/it]


Processed 49 questions successfully
Failed on 1 questions: ['rag_for_knowledge_intensive_tasks_q016']...


In [26]:
# Workaround: Mock the vertexai import to prevent import errors
import sys
from unittest.mock import MagicMock

sys.modules['google.cloud'] = MagicMock()
sys.modules['google.cloud.aiplatform'] = MagicMock()
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()

# Create Ragas dataset
from ragas import EvaluationDataset

print("Creating Ragas evaluation dataset...")
evaluation_dataset = EvaluationDataset.from_list(evaluation_data)

print(f"Dataset created with {len(evaluation_dataset)} samples")

Creating Ragas evaluation dataset...
Dataset created with 49 samples


In [28]:
# Workaround: Mock the vertexai import to prevent import errors
import sys
from unittest.mock import MagicMock

sys.modules['google.cloud'] = MagicMock()
sys.modules['google.cloud.aiplatform'] = MagicMock()
sys.modules['langchain_community.chat_models'] = MagicMock()
sys.modules['langchain_community.chat_models.vertexai'] = MagicMock()

# Run Ragas evaluation
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas.run_config import RunConfig

print("Initializing Ragas evaluator with Groq LLM...")
evaluator_llm = LangchainLLMWrapper(llm)

print("Running Ragas evaluation on 3 metrics...")
print("- LLMContextRecall")
print("- Faithfulness")
print("- FactualCorrectness")
print("\nThis may take several minutes...\n")

# Configure RunConfig to reduce parallelization and increase timeout
run_config = RunConfig(
    max_workers=1,  # Sequential evaluation (no parallelization)
    timeout=120,    # 120 seconds per evaluation
)

results = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
    ],
    llm=evaluator_llm,
    run_config=run_config,
)

print("\n" + "="*80)
print("EVALUATION RESULTS")
print("="*80)
print(results)

/tmp/ipykernel_2964/185310560.py:13: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
/tmp/ipykernel_2964/185310560.py:13: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
/tmp/ipykernel_2964/185310560.py:13: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import LLMContextRecall, Faithfulness, Factua

Initializing Ragas evaluator with Groq LLM...
Running Ragas evaluation on 3 metrics...
- LLMContextRecall
- Faithfulness
- FactualCorrectness

This may take several minutes...



Evaluating: 100%|██████████| 147/147 [1:56:52<00:00, 47.70s/it] 



EVALUATION RESULTS
{'context_recall': 0.7139, 'faithfulness': 0.6209, 'factual_correctness(mode=f1)': 0.3473}


In [31]:
import pandas as pd

# Convert results to DataFrame for analysis
results_df = results.to_pandas()

print("\nDetailed Metrics Summary:")
print("-" * 80)
print(f"Total samples evaluated: {len(results_df)}")
print(f"\nMetric Scores:")
print(f"  Context Recall average:     {results_df['context_recall'].mean():.4f}")
print(f"  Faithfulness average:       {results_df['faithfulness'].mean():.4f}")
print(f"  Factual Correctness average: {results_df['factual_correctness(mode=f1)'].mean():.4f}")

print(f"\nMetric Ranges:")
print(f"  Context Recall:     [{results_df['context_recall'].min():.4f}, {results_df['context_recall'].max():.4f}]")
print(f"  Faithfulness:       [{results_df['faithfulness'].min():.4f}, {results_df['faithfulness'].max():.4f}]")
print(f"  Factual Correctness: [{results_df['factual_correctness(mode=f1)'].min():.4f}, {results_df['factual_correctness(mode=f1)'].max():.4f}]")

print(f"\nSample Results (first 5):")
print(results_df[['user_input', 'context_recall', 'faithfulness', 'factual_correctness(mode=f1)']].head())

# Save results to CSV for reference
output_path = "../results/evaluation_results_50samples.csv"
results_df.to_csv(output_path, index=False)
print(f"\nResults saved to {output_path}")


Detailed Metrics Summary:
--------------------------------------------------------------------------------
Total samples evaluated: 49

Metric Scores:
  Context Recall average:     0.7139
  Faithfulness average:       0.6209
  Factual Correctness average: 0.3473

Metric Ranges:
  Context Recall:     [0.0000, 1.0000]
  Faithfulness:       [0.0000, 1.0000]
  Factual Correctness: [0.0000, 1.0000]

Sample Results (first 5):
                                          user_input  context_recall  \
0  How does the GANsformer update the input eleme...             0.0   
1  How is the Transformers library designed to se...             1.0   
2  How does intra-subject learning compare to cro...             1.0   
3  In the on-device computation experiment, how m...             1.0   
4  On what corpora was the BioBERT model trained,...             0.0   

   faithfulness  factual_correctness(mode=f1)  
0      0.571429                          0.00  
1      1.000000                          0.95 

In [30]:
# First, check what columns are actually in the results
print("Available columns in results_df:")
print(results_df.columns.tolist())
print("\nDataFrame shape:", results_df.shape)
print("\nFirst few rows:")
print(results_df.head())

Available columns in results_df:
['user_input', 'retrieved_contexts', 'response', 'reference', 'context_recall', 'faithfulness', 'factual_correctness(mode=f1)']

DataFrame shape: (49, 7)

First few rows:
                                          user_input  \
0  How does the GANsformer update the input eleme...   
1  How is the Transformers library designed to se...   
2  How does intra-subject learning compare to cro...   
3  In the on-device computation experiment, how m...   
4  On what corpora was the BioBERT model trained,...   

                                  retrieved_contexts  \
0  [the case of CLEVR producing high-quality imag...   
1  [NLP, it is important for these models to be a...   
2  [the target application area. Naturally, the d...   
3  [of parameters of each model along with the in...   
4  [Source:[36]Unlike SciBERT, the authors of thi...   

                                            response  \
0  According to the context, the Self-Attention o...   
1  Accordi